Lets check how can dates can be made to return same format

Below will automatically convert a date written as text (a string) into a structured date object.

• if **isinstance**(v, str):
This checks the data type of the variable v. It asks: "Is the value currently stored in v a string (text)?" If v is already a date object, a number, or None, this condition becomes False and the code skips it.

• **date_parser.parse**(v)
If v is a string (like "2026-11-09" or "Today"), this calls a parsing function (usually from a library like dateutil.parser). The parser intelligently reads the text, figures out the format, and turns it into a Python datetime object.

• **return**
This immediately sends the newly created datetime object back to whatever called the function.

In [ ]:
from dateutil import parser as date_parser
from datetime import datetime

def process_birthday(v):
    # If it's a string, convert it to a datetime object first
    if isinstance(v, str):
        return date_parser.parse(v)

    # If it's already a datetime object, just return it as-is
    return v

# Both of these will successfully return the exact same datetime object:
date1 = process_birthday("2026-11-09")
date2 = process_birthday(datetime(2026, 11, 9))


In [ ]:
print(date1)

2026-11-09 00:00:00


In [ ]:
print(date2)

2026-11-09 00:00:00


Data validation using pydantic

In [ ]:
from pydantic import BaseModel, ValidationError, validator , NonNegativeInt

Lets define a **pydantic class** , for the kind of structured output we would like to expect and validate

In [ ]:
class Person(BaseModel):
  name : str
  age  : NonNegativeInt
  city : str

Lets check
1. JSON schema , the format it will generate
2. Validate if the input is not in that type - whether it will throw error or not

In [ ]:
print(Person.model_json_schema())
# Check the defualt values that model generates

{'properties': {'name': {'title': 'Name', 'type': 'string'}, 'age': {'minimum': 0, 'title': 'Age', 'type': 'integer'}, 'city': {'title': 'City', 'type': 'string'}}, 'required': ['name', 'age', 'city'], 'title': 'Person', 'type': 'object'}


In [ ]:
# person1 = Person(name = "Michael", age = "35sth" , city = "Berlin") - this will get error
person1 = Person(name = "Michael", age = "35" , city = "Berlin")

Model with optional Field

In [ ]:
from typing import Optional

In [ ]:
class Car(BaseModel):
  carmodel_name : str
  model_year : NonNegativeInt
  milage : Optional[float] = None # Optional


In [ ]:
vw_taigun = Car(carmodel_name = "VW-Taigun", model_year=2023 , milage= 10.2)
vw_taigun

Car(carmodel_name='VW-Taigun', model_year=2023, milage=10.2)

Model with **List** value

In [ ]:
from typing import List

In [ ]:
class Classroom(BaseModel):
  grade : str
  room_number : NonNegativeInt
  students : List[str]
  capacity : NonNegativeInt

In [ ]:
try:
  invalid_clsrm = Classroom(grade="V-A",room_number="A501",students= ["Akash","Bikash",P231])
except ValueError as e:
  print(e)

Pydantic **fields** : Customization and Constraints
Field() function in pydantic enhances model fields beyond basic type checking by allowing you to specify validation rules , default values , aliases and more

In [ ]:
from pydantic import Field

In [ ]:
class AdvanceClsrm (BaseModel):
  grade : str          = Field(..., min_length=1, max_length=6)
  room_number : int    = Field(..., gt=0, le = 12)
  students : List[str] = Field(..., min_length=1)
  capacity : Optional[int]        =  Field(gt=0, le = 25 )

In [ ]:
#balidate actual

myAdvCls = AdvanceClsrm( grade= "V-B", room_number= 11 , students= ["Akash", "Bikash", "Prakash"] , capacity= 24) # test with any invalid value like room_number = 14
print(myAdvCls)

grade='V-B' room_number=11 students=['Akash', 'Bikash', 'Prakash'] capacity=24


The **@field_validator** in Pydantic v2 is a **decorator** used to define custom validation or data-cleaning logic for specific fields inside a BaseModel.
By default, it runs after Pydantic has performed its standard type conversion and basic checks (mode='after'). **You can also run it before standard type checking (mode='before')** to preprocess raw input.

Key Rules

• Must be decorated with @classmethod.
• Must return the validated (and optionally modified) value, or raise a ValueError if the data is invalid.
Receives the field value (and optionally info: ValidationInfo for metadata or checking other fields)

**Example 1: After-Validation (Custom Business Logic & Error Raising)**

In [ ]:
from pydantic import BaseModel, field_validator

class User(BaseModel):
    username: str

    @field_validator("username")
    @classmethod
    def validate_username(cls, v: str) -> str:
        if " " in v:
            raise ValueError("Username cannot contain spaces")
        if len(v) < 3:
            raise ValueError("Username must be at least 3 characters long")
        return v

# Valid
user = User(username="alice_dev")

# Invalid -> raises ValueError
# user = User(username="al ice")



**Example 2: Data Mutation / Cleaning**

Validators can modify or transform the input data before it gets stored in the model instance.

In [ ]:
from pydantic import BaseModel, field_validator

class Profile(BaseModel):
    email: str

    @field_validator("email")
    @classmethod
    def clean_email(cls, v: str) -> str:
        # Strip whitespace and convert to lowercase
        return v.strip().lower()

p = Profile(email="  John.Doe@Example.COM  ")
print(p.email)  # Output: "john.doe@example.com"


john.doe@example.com


In [ ]:
# inconsistent formats ("January 25, 2016" vs "2016-01-25"), so we add a
# validator to normalize whatever string comes back before Pydantic parses it
# into a datetime — this is what prevents the ValueError
class Invoice(BaseModel):
    """A representation of information from an invoice."""

    vendor: str
    invoice_date: datetime
    due_date: datetime
    invoice_number: str
    items: List[str]

    # before validating the field, call this function
    @field_validator("invoice_date", "due_date", mode="before") #tells Pydantic when to run your validation function.
    @classmethod #tells Python how to call that function
    def normalize_date(cls, v):
        if isinstance(v, str):
            return date_parser.parse(v)
        return v

#dt_x = "2026-11-10"
#dt_y = datetime(2026, 11, 10)
my_inv = Invoice(vendor="xyz ltd",invoice_date= "2026-11-10",due_date= datetime(2026,11,24),invoice_number= "65432B",items= ["item1","item2"])
print(my_inv)

vendor='xyz ltd' invoice_date=datetime.datetime(2026, 11, 10, 0, 0) due_date=datetime.datetime(2026, 11, 24, 0, 0) invoice_number='65432B' items=['item1', 'item2']
